In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, date_format


def create_gold_dim_date(Customer_tbl_df, Person_tbl_df):

    Customer_tbl_df = Customer_tbl_df.select("CustomerID", "AccountNumber", "TerritoryID", "PersonId")
    Person_tbl_df = Person_tbl_df.select("FirstName","LastName","PersonId")
    customer_tgt_df = Customer_tbl_df.join(Person_tbl_df, Customer_tbl_df.PersonId == Person_tbl_df.PersonId, "inner").drop(Person_tbl_df.PersonId).dropDuplicates(["CustomerID"]).withColumn("processed_timestamp", F.current_timestamp())
                                 
    return customer_tgt_df



if __name__ == "__main__":

    Customer_tbl = dbutils.widgets.get("Customer")
    Person_tbl = dbutils.widgets.get("Person")
    Customer_tbl_df = df = spark.read.table(Customer_tbl)
    Person_tbl_df = df = spark.read.table(Person_tbl)

    customer_tgt_df = create_gold_dim_date(Customer_tbl_df, Person_tbl_df)
    customer_gold_tbl = dbutils.widgets.get("Customer_gold_tbl")
    customer_tgt_df.write.mode("overwrite").format("delta").partitionBy("PersonId").saveAsTable(customer_gold_tbl)